# MedSAM2: segment a 3D CT by prompting a single slice

A walkthrough of the method end to end. SAM 2 propagates a prompt across video frames with
memory attention; a CT volume has the same structure, so one box on one slice can be
tracked through the whole stack.

**Runtime → Change runtime type → T4 GPU** before running anything.

- Repo: https://github.com/bowang-lab/MedSAM2
- Weights: https://huggingface.co/wanglab/MedSAM2 (research and education only)
- Paper: https://arxiv.org/abs/2504.03600

## 1 · Environment

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print(torch.cuda.get_device_name(0))

In [ ]:
%%capture
!pip install -q git+https://github.com/rekalantar/medsam2-3d-ct.git
!pip install -q SimpleITK imageio huggingface_hub

In [ ]:
%%capture
!git clone -q https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
%cd /content/MedSAM2
!pip install -q -e ".[dev]"
!bash download.sh

In [ ]:
import os

# confirm the checkpoint actually landed before going further
ckpt = "/content/MedSAM2/checkpoints/MedSAM2_latest.pt"
assert os.path.exists(ckpt), "download.sh did not produce MedSAM2_latest.pt"
print(f"{os.path.getsize(ckpt) / 1e6:.0f} MB")

## 2 · Get a volume

Two options. Either upload your own `.nii.gz`, or pull a case from the MedSAM2 demo
dataset. The dataset is large, so list it first and fetch a single file rather than
calling `load_dataset`, which would pull tens of gigabytes.

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("wanglab/CT_DeepLesion-MedSAM2", repo_type="dataset")
print(len(files), "files")
for f in files[:20]:
    print(' ', f)

In [ ]:
# Pick one from the listing above, or set VOLUME_PATH to your own upload.
from huggingface_hub import hf_hub_download

VOLUME_PATH = None  # e.g. '/content/my_scan.nii.gz'

if VOLUME_PATH is None:
    VOLUME_PATH = hf_hub_download(
        "wanglab/CT_DeepLesion-MedSAM2",
        filename="<paste a filename from the listing>",
        repo_type="dataset",
    )
print(VOLUME_PATH)

## 3 · Load and window

The step people skip. CT spans several thousand Hounsfield units; the network wants
8-bit. Min-maxing the full range collapses soft tissue into a handful of grey levels.
W400/L40 is the abdominal soft-tissue window.

In [ ]:
from medsam2_ct import load_volume, window_hu

volume_hu, spacing = load_volume(VOLUME_PATH)   # (z, y, x), spacing in mm
volume = window_hu(volume_hu, width=400, level=40)

print(f"shape {volume.shape}  spacing {spacing}  range {volume.min()}-{volume.max()}")

## 4 · Choose the key slice and draw the box

Pick the slice where the lesion is clearest — usually its largest cross-section.

In [ ]:
import matplotlib.pyplot as plt

KEY_SLICE = volume.shape[0] // 2   # adjust after looking

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, i in zip(axes, [KEY_SLICE - 10, KEY_SLICE, KEY_SLICE + 10]):
    ax.imshow(volume[i], cmap="gray")
    ax.set_title(f"slice {i}")
    ax.axis("off")
plt.tight_layout()

In [ ]:
# [x_min, y_min, x_max, y_max] on KEY_SLICE. Read the coordinates off the plot above.
BOX = [180, 180, 260, 260]

import matplotlib.patches as patches
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(volume[KEY_SLICE], cmap="gray")
ax.add_patch(patches.Rectangle(
    (BOX[0], BOX[1]), BOX[2] - BOX[0], BOX[3] - BOX[1],
    edgecolor="#FF6B5B", facecolor="none", linewidth=2))
ax.set_title(f"prompt on slice {KEY_SLICE}")
ax.axis("off")

## 5 · One box, whole volume

`segment_volume` propagates forward and backward. The key slice sits mid-lesion, so
forward-only would capture roughly half of it.

In [ ]:
from medsam2_ct import build_predictor, segment_volume, largest_component

predictor = build_predictor(
    config="configs/sam2.1_hiera_t512.yaml",
    checkpoint=ckpt,
)

masks = segment_volume(predictor, volume, BOX, KEY_SLICE)
print(f"raw: {masks.sum():,} voxels across {(masks.any(axis=(1,2))).sum()} slices")

masks = largest_component(masks)
print(f"largest component: {masks.sum():,} voxels")

## 6 · The payoff figure

Watch it rather than trusting a number. Propagation failures are obvious to the eye
and invisible in a mean. Keep the GIF under 8 MB or Medium rejects it.

In [ ]:
from medsam2_ct import save_gif

z = masks.any(axis=(1, 2)).nonzero()[0]
lo, hi = max(0, z.min() - 4), min(len(volume), z.max() + 5)

save_gif(volume[lo:hi], masks[lo:hi], 'medsam2_propagation.gif', fps=8)

import os
print(f"{os.path.getsize('medsam2_propagation.gif') / 1e6:.1f} MB")

In [ ]:
from IPython.display import Image
Image('medsam2_propagation.gif')

## 7 · Check the claims the article makes

The post asserts three things. Verify each before publishing — if one doesn't hold on
your data, the honest section needs rewriting, not the result.

**Claim 1 — windowing changes the result.**

In [ ]:
import numpy as np

naive = ((volume_hu - volume_hu.min()) /
         (volume_hu.max() - volume_hu.min()) * 255).astype(np.uint8)

masks_naive = largest_component(segment_volume(predictor, naive, BOX, KEY_SLICE))

inter = (masks & masks_naive).sum()
dice = 2 * inter / (masks.sum() + masks_naive.sum())
print(f"windowed {masks.sum():,} vs naive {masks_naive.sum():,} voxels  |  Dice {dice:.3f}")

**Claim 2 — backward propagation matters.** Forward-only should lose roughly half.

In [ ]:
state = predictor.init_state(volume, volume.shape[1], volume.shape[2])
predictor.add_new_points_or_box(
    inference_state=state, frame_idx=KEY_SLICE, obj_id=1,
    box=np.asarray(BOX, dtype=np.float32),
)

fwd = np.zeros_like(masks)
for idx, _ids, logits in predictor.propagate_in_video(state):
    fwd[idx] = (logits[0] > 0).cpu().numpy().squeeze()

print(f"bidirectional {masks.sum():,}  |  forward-only {fwd.sum():,}"
      f"  ({fwd.sum() / masks.sum():.0%})")

**Claim 3 — accuracy decays with distance from the prompt.**

Per-slice area against distance from the key slice. If it holds flat all the way to
both extremes, the article's honest section is wrong and should say so instead.

In [ ]:
areas = masks.sum(axis=(1, 2))
z = areas.nonzero()[0]

plt.figure(figsize=(8, 3.5))
plt.plot(z - KEY_SLICE, areas[z], color="#3FC1C9", linewidth=2)
plt.axvline(0, color="#FF6B5B", linestyle="--", label="prompted slice")
plt.xlabel("slices from prompt"); plt.ylabel("mask area (voxels)")
plt.legend(); plt.tight_layout()

---

Project: https://github.com/rekalantar/medsam2-3d-ct